# Lab 5A: Sine-Sweep Frequency Response

Run the cells in order. The MicroPython program remains active and waits for
commands after the serial connection is established.


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
from serial import Serial
from serial.tools import list_ports
import time
import numpy as np
from matplotlib import pyplot as plt


## 1. Open the Serial Connection

Keep the USB cable connected to the Shoe of Brian. In Thonny, record the serial
port shown for `MicroPython (generic)`, then switch the interpreter to
`Local Python 3` so that Jupyter can use the same port.

List the available ports and identify that Shoe USB port.


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
for port in list_ports.comports():
    print(port.device, port.description)


Set `SERIAL_PORT` to the Shoe USB port found above. This is the only value
to change in the serial-connection setup.


In [ ]:
# TODO: Enter the Shoe USB serial port.
SERIAL_PORT = "COM3"

# PROVIDED VALUE — DO NOT MODIFY.
BAUDRATE = 115200


Open the serial port, restart the MicroPython program, and wait until
`main.py` prints `READY LAB5A_SERIAL_V1`.

**Provided code — do not modify this cell.**


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
ser = Serial(SERIAL_PORT, baudrate=BAUDRATE, timeout=0.2)
time.sleep(0.3)
ser.reset_input_buffer()

# Ctrl-B returns to the normal REPL, Ctrl-C stops a running program,
# and Ctrl-D soft-resets MicroPython so main.py runs again.
ser.write(b"\x02")
time.sleep(0.1)
ser.write(b"\x03")
time.sleep(0.1)
ser.write(b"\x04")

deadline = time.time() + 5
ready = False

while time.time() < deadline and ready == False:
    line = ser.readline().decode().strip()

    if line == "READY LAB5A_SERIAL_V1":
        ready = True
        print(line)

if ready == False:
    print("ERR did not receive READY. Check that main.py is uploaded and has no errors.")
else:
    print("Serial port is open and main.py is running.")


## 2. Define `run_command()`

`run_command()` sends one complete command through the open serial connection
and reads the response through `END`. After responding, the MicroPython program
immediately waits for another command.

**Provided code — do not modify this cell.**


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
def run_command(command, timeout=45):
    lines = []

    ser.write((command + "\r\n").encode())

    deadline = time.time() + timeout
    finished = False

    while time.time() < deadline and finished == False:
        current_line = ser.readline().decode().strip()

        if current_line != "":
            lines.append(current_line)

            if current_line == "END":
                finished = True
            elif current_line.startswith("DATA,") == False:
                print(current_line)

    if finished == False:
        print("ERR no END before timeout")

    return lines


## 3. Set the Sine-Sweep Parameters

Choose the frequency range specified for your pendulum. The sample rate is the
number of measurements collected per second. Use a unique filename whenever you
want to preserve a run.


In [ ]:
# TODO: Select the frequency range for your pendulum and a data filename.
# Short pendulum: 3–8 Hz. Long pendulum: 2–6 Hz.
AMPLITUDE_PERCENT = 100
F_START_HZ = 3.0
F_END_HZ = 8.0
SWEEP_TIME_S = 30
SAMPLE_RATE_HZ = 100
DATA_FILENAME = "sweep_data.csv"


## 4. Run One Sine-Sweep Experiment

The provided cell constructs a `RUN_SWEEP` command from the parameters above.
Do not edit the command phrase or communication code.


In [ ]:
# PROVIDED COMMAND CODE — DO NOT MODIFY THIS CELL.
response_lines = run_command(
    (
        f"RUN_SWEEP {AMPLITUDE_PERCENT} {F_START_HZ} {F_END_HZ} "
        f"{SWEEP_TIME_S} {SAMPLE_RATE_HZ}"
    ),
    timeout=SWEEP_TIME_S + 10,
)


## 5. Save the Returned Data

Extract the `DATA` lines from the most recent response and save the time,
voltage-percentage input, and angular-position output to the selected CSV file.

**Provided code — do not modify this cell.**


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
data_lines = [
    line[len("DATA,"):]
    for line in response_lines
    if line.startswith("DATA,")
]

if len(data_lines) == 0:
    print("ERR no data were returned by the microcontroller")
else:
    with open(DATA_FILENAME, "w") as data_file:
        for data_line in data_lines:
            data_file.write(data_line + "\n")

    print(f"Saved {len(data_lines)} samples to {DATA_FILENAME}.")


## 6. Inspect the Time-Domain Signals

Load the saved CSV file and assign its three columns to clearly named arrays.


In [ ]:
data = np.genfromtxt(DATA_FILENAME, delimiter=",")

times_s = data[:, 0]
# TODO: Store column 1 as voltage_percent.
# TODO: Store column 2 as position_rad.


Plot the sine-sweep input and measured position versus time.

**Provided code — do not modify this cell.**


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

axes[0].plot(times_s, voltage_percent)
axes[0].set_ylabel("Voltage command [%]")
axes[0].set_title("Sine-Sweep Input")
axes[0].grid(True)

axes[1].plot(times_s, position_rad)
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Position [rad]")
axes[1].set_title("Measured Pendulum Position")
axes[1].grid(True)

fig.tight_layout()
plt.show()


## 7. Calculate the Experimental Frequency Response

The function below shows how NumPy transforms the time-domain input and output
into a frequency response. Read through the calculation, but do not modify it.


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
def fft_for_bode(times, input_data, output_data):
    """Convert equally sampled input and output data into Bode-plot arrays.

    Args:
        times: Sample times in seconds.
        input_data: Excitation applied to the system input.
        output_data: Measured system output.

    Returns:
        A tuple (frequencies, magnitude_db, phase_deg).
    """
    # Determine the constant sample period from the time data.
    sample_period_s = times[1] - times[0]
    frequencies = np.fft.rfftfreq(len(times), d=sample_period_s)

    # Transform the input and output signals into complex frequency components.
    input_fft = np.fft.rfft(input_data)
    output_fft = np.fft.rfft(output_data)

    # Divide output by input at each frequency to estimate the response.
    response = output_fft / input_fft

    magnitude = np.abs(response)
    magnitude_db = 20 * np.log10(magnitude)
    phase_deg = np.angle(response, deg=True)

    return frequencies, magnitude_db, phase_deg


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
frequencies_hz, magnitude_db, phase_deg = fft_for_bode(
    times_s,
    voltage_percent,
    position_rad,
)


## 8. Plot the Experimental Bode Diagram

Complete the cell below using two semilogarithmic-x subplots. Plot magnitude in
decibels above phase in degrees, label the axes, and show the experimental data
as points. Do not discard the data outside the commanded sweep range yet; you
will interpret its reliability in the lab deliverables.


In [ ]:
# TODO: Plot the experimental magnitude and phase in two semilog-x subplots.
# Use frequencies_hz, magnitude_db, and phase_deg. Exclude only the 0 Hz point
# because zero cannot be displayed on a logarithmic frequency axis.


## 9. Fit a Second-Order Model

Adjust the three model parameters to fit the experimental Bode diagram. Enter
the natural frequency in Hz; the provided calculation converts it to rad/s.


In [ ]:
# TODO: Adjust only these three values to fit your experimental data.
K_SS = 0.01
NATURAL_FREQUENCY_HZ = 1.0
DAMPING_RATIO = 0.10

# PROVIDED MODEL CALCULATION — DO NOT MODIFY BELOW THIS LINE.
model_frequencies_hz = np.logspace(-1, 1, 1000)
s = 1.0j * 2 * np.pi * model_frequencies_hz
omega_n_rad_s = 2 * np.pi * NATURAL_FREQUENCY_HZ

G = (
    K_SS * omega_n_rad_s ** 2
    / (
        s ** 2
        + 2 * DAMPING_RATIO * omega_n_rad_s * s
        + omega_n_rad_s ** 2
    )
)

model_magnitude_db = 20 * np.log10(np.abs(G))
model_phase_deg = np.angle(G, deg=True)


Overlay the fitted model as solid lines on the experimental magnitude and
phase points. Include labels and a legend so the two results are distinguishable.


In [ ]:
# TODO: Plot the experimental Bode points and overlay the fitted model.
# Use model_frequencies_hz, model_magnitude_db, and model_phase_deg for the model.


## 10. Close the Serial Connection

After all tests are complete, close the serial port. You may then switch Thonny
back to `MicroPython (generic)`.


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
ser.close()
